<!-- NB_DOC_INTRO_V1 -->
# EDA — Analyses multivariees (PCA, MCA, t-SNE, UMAP)

> 📚 **Doc thematique** : [docs/02_EDA.md](docs/02_EDA.md) (EDA & Visualisation)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Reduction de dimension** = visualiser des donnees > 3D, identifier les axes de variance principaux, decorreler les features avant ML. Ce notebook execute les methodes principales sur **iris** (toy) et compare leurs trade-offs : lineaires (PCA, LDA, MCA, FAMD), non-lineaires (t-SNE, UMAP, MDS).

Cas d'usage : **PCA** pour ML pipeline (decorrelation + variance), **t-SNE/UMAP** pour visualisation de clusters, **MCA/FAMD** quand features categorielles.

## Plan

1. PCA (linear, variance maximale)
2. PCA biplot (interpretation des axes)
3. LDA (supervise, separation classes)
4. t-SNE (non-lineaire, preservation voisinage local)
5. UMAP (non-lineaire, scale)
6. MDS (preservation distances)
7. MCA / FAMD (pour quali ou mixte)
8. Comparatif methodes (tableau)
9. Pieges et anti-patterns
10. References


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE, MDS
from sklearn.datasets import load_iris, load_digits
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# Iris (4 features, 3 classes)
iris = load_iris(as_frame=True)
X, y = iris.data, iris.target
class_names = iris.target_names
print(f"X shape : {X.shape}, classes : {class_names}")

## 1. PCA — Analyse en Composantes Principales

Decorrele les features et **maximise la variance** projetee. Lineaire. Premier choix par defaut.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_scaled)

# Variance expliquee
print(f"Variance expliquee par composante : {pca.explained_variance_ratio_.round(3)}")
print(f"Cumulee : {pca.explained_variance_ratio_.cumsum().round(3)}")
print(f"\n=> PC1+PC2 capture {pca.explained_variance_ratio_[:2].sum():.1%} de la variance")

In [ ]:
# Scree plot (eboulis) + cumulative
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(1, 5), pca.explained_variance_ratio_, color="steelblue")
axes[0].set(xlabel="Component", ylabel="Variance ratio", title="Scree plot (eboulis)")

axes[1].plot(range(1, 5), pca.explained_variance_ratio_.cumsum(), "o-", color="C3")
axes[1].axhline(0.95, color="grey", ls="--", label="95% cumulee")
axes[1].set(xlabel="Component", ylabel="Variance cumulee", title="Cumulative")
axes[1].legend()
plt.tight_layout()
plt.show()

## 2. PCA biplot — interpretation des axes

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Scatter individus
for cls, name in enumerate(class_names):
    mask = y == cls
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], label=name, s=50, alpha=0.7)

# Fleches des features (loadings)
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
for i, feature in enumerate(X.columns):
    ax.arrow(0, 0, loadings[i, 0]*3, loadings[i, 1]*3,
             color="red", alpha=0.8, head_width=0.1)
    ax.text(loadings[i, 0]*3.3, loadings[i, 1]*3.3, feature, color="red", fontsize=11)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("Biplot PCA — individus + variables")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretation biplot** : les **fleches** sont les variables ; leur direction = axe de croissance ; leur longueur = qualite de representation sur PC1×PC2. Les **points** sont les individus dans l'espace reduit.

## 3. LDA — Linear Discriminant Analysis

**Supervise** : maximise la separation entre classes (au contraire de PCA qui ignore les labels).

In [ ]:
lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X_scaled, y)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PCA (non supervise)
for cls, name in enumerate(class_names):
    mask = y == cls
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], label=name, s=50, alpha=0.7)
axes[0].set_title("PCA (non supervise)")
axes[0].legend()

# LDA (supervise)
for cls, name in enumerate(class_names):
    mask = y == cls
    axes[1].scatter(X_lda[mask, 0], X_lda[mask, 1], label=name, s=50, alpha=0.7)
axes[1].set_title("LDA (supervise — separe les classes)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. t-SNE — non-lineaire, preservation des voisinages locaux

Tres bon pour **visualiser des clusters**. Lent (O(N²)) au-dela de 10k points. Stochastique : resultat depend de `random_state` et `perplexity`.

In [ ]:
# t-SNE sur iris (vite, 150 points)
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, init="pca")
X_tsne = tsne.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
for cls, name in enumerate(class_names):
    mask = y == cls
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=name, s=50, alpha=0.7)
ax.set_title(f"t-SNE (perplexity=30)")
ax.legend()
plt.show()

### `perplexity` : effet majeur

`perplexity` ≈ **nombre de voisins** consideres pour chaque point. Petit (5-10) = focus tres local. Grand (50-100) = vue globale.

> 🎯 Toujours essayer **plusieurs valeurs** avant de conclure.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, perp in zip(axes, [5, 15, 30, 50]):
    t = TSNE(n_components=2, perplexity=perp, random_state=SEED, init="pca")
    Xt = t.fit_transform(X_scaled)
    for cls in range(3):
        ax.scatter(Xt[y == cls, 0], Xt[y == cls, 1], s=40, alpha=0.7)
    ax.set_title(f"perplexity={perp}")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

## 5. UMAP — non-lineaire, scale

Plus rapide que t-SNE, preserve mieux la **structure globale**. Standard de fait en 2024-2025.

In [ ]:
try:
    import umap
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED)
    X_umap = reducer.fit_transform(X_scaled)

    fig, ax = plt.subplots(figsize=(8, 6))
    for cls, name in enumerate(class_names):
        mask = y == cls
        ax.scatter(X_umap[mask, 0], X_umap[mask, 1], label=name, s=50, alpha=0.7)
    ax.set_title("UMAP")
    ax.legend()
    plt.show()
except ImportError:
    print("UMAP non installe : pip install umap-learn")
    print("(t-SNE fait deja le job ici, sur iris)")

## 6. MDS — Multidimensional Scaling

Preserve les **distances** (a la difference de t-SNE qui preserve les voisinages).

In [ ]:
mds = MDS(n_components=2, random_state=SEED, n_init=4)
X_mds = mds.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
for cls, name in enumerate(class_names):
    mask = y == cls
    ax.scatter(X_mds[mask, 0], X_mds[mask, 1], label=name, s=50, alpha=0.7)
ax.set_title(f"MDS (stress={mds.stress_:.2f})")
ax.legend()
plt.show()

## 7. MCA / FAMD — pour quali ou mixte

**MCA** (Multiple Correspondence Analysis) = analogue PCA mais pour **variables categorielles**.
**FAMD** (Factor Analysis of Mixed Data) = pour datasets **mixtes** (quanti + quali).

```python
# pip install prince
import prince

# MCA sur dataset categoriel
mca = prince.MCA(n_components=2, random_state=42)
mca.fit(df_categorical)
mca.plot_coordinates(df_categorical)

# FAMD pour mixte
famd = prince.FAMD(n_components=2, random_state=42)
famd.fit(df_mixed)
```


## 8. Comparatif methodes

| Methode | Type | Vitesse | Preserve | Cas d'usage |
|---|---|---|---|---|
| **PCA** | Lineaire, non-supervise | ★★★★★ | Variance | Pipeline ML, decorrelation, 1ère viz |
| **LDA** | Lineaire, supervise | ★★★★ | Separation classes | Quand on a y et on veut visualiser |
| **MCA** | Lineaire, non-sup | ★★★★ | Chi² | Dataset purement categoriel |
| **FAMD** | Lineaire, non-sup | ★★★ | Variance + Chi² | Dataset mixte |
| **MDS** | Non-lineaire | ★★ | Distances pairwise | Quand on a juste une matrice de distances |
| **t-SNE** | Non-lineaire | ★ | Voisinages locaux | Viz de clusters en haute dim |
| **UMAP** | Non-lineaire | ★★★ | Local + global | Standard de fait 2024-2025 |
| **PaCMAP / TriMap** | Non-lineaire | ★★★ | Structure | Alternatives recentes |

## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Faire PCA **sans standardiser** | Variables a grande echelle dominent — `StandardScaler` d'abord |
| Interpreter t-SNE comme une vraie projection 2D | t-SNE deforme les distances, ne pas mesurer dessus |
| `perplexity` par defaut t-SNE | Toujours essayer 5, 15, 30, 50 |
| Garder toutes les PCs sans logique | Cumulee 95% ou critere Kaiser (eigval > 1) |
| LDA quand `n_features > n_samples` | Mal conditionne — regulariser ou utiliser PCA d'abord |
| MCA sur variables **ordinales** | Perd l'ordre — encoder en numerique d'abord |
| UMAP sans fixer `random_state` | Resultats non reproductibles |
| Comparer 2 t-SNE entre eux | Stochastique : pas comparable directement |

## 10. References

- **PCA** : https://scikit-learn.org/stable/modules/decomposition.html#pca
- **LDA** : https://scikit-learn.org/stable/modules/lda_qda.html
- **t-SNE** : van der Maaten & Hinton (2008). Doc : https://scikit-learn.org/stable/modules/manifold.html#t-sne
- **UMAP** : McInnes et al. (2018). https://umap-learn.readthedocs.io/
- **prince** (MCA/FAMD) : https://github.com/MaxHalford/prince
- *Understanding UMAP* : https://pair-code.github.io/understanding-umap/

### Voir aussi (collection)
- [EDA_Stats_Analyse_Desc_Visual.ipynb](EDA_Stats_Analyse_Desc_Visual.ipynb)
- [EDA_Visualisation_Introduction.ipynb](EDA_Visualisation_Introduction.ipynb)
- [ML_Explication_Feature_Importance_Selection.ipynb](ML_Explication_Feature_Importance_Selection.ipynb)
